In [1]:
pip install torch transformers

In [3]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model():

    print("Loading AI assistant... Please wait.")
    model_name = "microsoft/DialoGPT-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    return tokenizer, model

def generate_response(user_input, tokenizer, model, chat_history_ids):
    """
    Generates a response using the transformer model while maintaining
    conversational context.
    """
    # Encode the new user input and append the eos_token
    new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append the new user input tokens to the chat history
    bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if chat_history_ids is not None else new_user_input_ids

    # Generate a response (max_length keeps the response concise)
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=100,
        top_p=0.7,
        temperature=0.8
    )

    # Decode only the last generated tokens (the bot's response)
    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    return response, chat_history_ids

def run_chatbot():
    # 1. Model Loading
    tokenizer, model = load_model()

    chat_history_ids = None
    print("\n--- Chatbot Started ---")
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    while True:
        # 2. User Input Handling
        user_input = input("User: ")

        # 5. Exit Condition
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # 3. Response Generation & 4. Continuous Conversation
        try:
            response, chat_history_ids = generate_response(
                user_input, tokenizer, model, chat_history_ids
            )

            # Display Output
            print(f"Chatbot: {response}")

        except Exception as e:
            print("Chatbot: I encountered a small error. Could you repeat that?")

if __name__ == "__main__":
    run_chatbot()

Loading AI assistant... Please wait.


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Chatbot Started ---
Chatbot: Hello! I am your AI assistant. How can I help you today?
User: exit
Chatbot: Goodbye! Have a great day.
